# FitAI — Food Classification v4
**ConvNeXt-Tiny · Class-Weighted Loss · MixUp · 3-Stage Fine-Tuning · TTA · Confidence Threshold**

**Expected training time on T4:** ~45–55 min

## Step 0: Mount Drive & Install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/FitAI/models_v4'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Checkpoint dir: {SAVE_DIR}')

In [ ]:
!pip install -q timm torchmetrics scikit-learn seaborn
print('Dependencies installed')

## Step 1: Build Merged Dataset

Merges your self-collected photos + Kaggle sweets dataset + Food-101 (if available).

Run this every Colab session — it rebuilds `/content/mess_dataset` from Drive.

In [ ]:
import os, shutil, random
import numpy as np

# ── Paths ──────────────────────────────────────────────────────────────────────
YOUR_PHOTOS        = '/content/drive/MyDrive/FitAI/FitAI_ImageDataset'
KAGGLE_ROOT        = '/content/drive/MyDrive/FitAI/kaggle_extracted'
FOOD101_ROOT       = '/content/drive/MyDrive/FitAI/food101_extracted'
FILTERED_DIR       = '/content/mess_dataset'
MAX_PER_CLASS      = 100   # hard cap — prevents Food-101 from dominating

# ── Kaggle sweets dataset → canonical class name ───────────────────────────────
SWEETS_MAP = {
    'aloo_gobi':            'aloo_gobi',
    'aloo_matar':           'aloo_matar',
    'biryani':              'biryani',
    'butter_chicken':       'butter_chicken',
    'chana_masala':         'chole',
    'chapati':              'chapati',
    'dal_makhani':          'dal_makhani',
    'dal_tadka':            'dal_fry',
    'gajar_ka_halwa':       'halwa',
    'gulab_jamun':          'gulab_jamun',
    'kadai_paneer':         'paneer_dish',
    'maach_jhol':           'odia_fish_curry',
    'naan':                 'naan',
    'palak_paneer':         'paneer_dish',
    'paneer_butter_masala': 'paneer_dish',
    'pithe':                'pitha',
    'poha':                 'poha',
    'rasgulla':             'rasgulla',
    'mixed_veg':            'mixed_veg',
}

# ── Food-101 → canonical class name ───────────────────────────────────────────
FOOD101_MAP = {
    'burger':        'burger',
    'pizza':         'pizza',
    'idli':          'idli',
    'dosa':          'dosa',
    'fried_rice':    'fried_rice',
    'chicken_curry': 'chicken_curry',
    'egg_omelette':  'egg_omelette',
    'puri':          'puri',
    'sambar':        'sambhar',
    'shawarma':      'shawarma',
    'biryani':       'biryani',
    'upma':          'upma',
    'pav_bhaji':     'pav_bhaji',
    'chole':         'chole',
    'naan':          'naan',
    'dal_fry':       'dal_fry',
    'mixed_veg':     'mixed_veg',
}

# ── Your self-collected folder names → canonical class name ────────────────────
YOUR_NAME_MAP = {
    'Fish':             'fish_masala',    # 12 imgs — KIIT specific
    'Biryani':          'biryani',
    'Manchurian':       'veg_manchurian',
    'Aloo_Gobi':        'aloo_gobi',
    'DOSA':             'dosa',
    'Puri':             'puri',
    'Egg':              'egg_omelette',
    'Burger':           'burger',
    'Pizza':            'pizza',
    'Shwarama':         'shawarma',
    'Ghoogni':          'ghuguni',
    'ghuguni_scraped':  'ghuguni',        # scraped — adds ~50 more
    'shawarma_scraped': 'shawarma',       # scraped — adds ~37 more
    'Vada':             'vada',
    'Rice':             'rice',
    'Roti':             'chapati',
    'Soyabean':         'soyabean',
    'Paneer_curry':     'paneer_dish',
    'MixVeg':           'mixed_veg',
    'Besan_Curry':      'besan_curry',
    'Chicken curry':    'chicken_curry',
    'Daal':             'dal_fry',
}

# Classes NOT in our 65-class target — drop them
DROP_CLASSES = {'besan_curry', 'soyabean', 'vada', 'salad', 'pakodi_manchurian'}

def copy_images(src, dst, prefix, limit=MAX_PER_CLASS):
    """Copy up to `limit` images from src → dst, skipping if already at cap."""
    os.makedirs(dst, exist_ok=True)
    imgs = [f for f in os.listdir(src)
            if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))]
    random.shuffle(imgs)
    existing = len([f for f in os.listdir(dst)
                    if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))])
    slots = max(0, limit - existing)
    for img in imgs[:slots]:
        shutil.copy(os.path.join(src, img),
                    os.path.join(dst, f'{prefix}_{img}'))
    return min(len(imgs), slots)

def find_root(base):
    """Find the actual image folder root inside an extracted zip."""
    if not os.path.exists(base):
        return None
    for root, dirs, files in os.walk(base):
        if len(dirs) > 5:
            return root
    return base

# ── Rebuild dataset from scratch ───────────────────────────────────────────────
if os.path.exists(FILTERED_DIR):
    shutil.rmtree(FILTERED_DIR)
random.seed(42)
stats = {}

# 1. Kaggle sweets dataset
sweets_root = find_root(KAGGLE_ROOT)
if sweets_root:
    print(f'Kaggle found: {sweets_root}')
    sweets_folders = {
        f.lower().replace(' ', '_').replace('-', '_'): f
        for f in os.listdir(sweets_root)
        if os.path.isdir(os.path.join(sweets_root, f))
    }
    for key, canonical in SWEETS_MAP.items():
        if canonical in DROP_CLASSES:
            continue
        matched = next((v for k, v in sweets_folders.items()
                        if key in k or k in key), None)
        if not matched:
            continue
        n = copy_images(os.path.join(sweets_root, matched),
                        os.path.join(FILTERED_DIR, canonical), 'kaggle')
        stats.setdefault(canonical, {'kaggle': 0, 'food101': 0, 'yours': 0})
        stats[canonical]['kaggle'] += n
else:
    print('Kaggle not found — skipping')

# 2. Food-101
food101_root = find_root(FOOD101_ROOT)
if food101_root:
    print(f'Food-101 found: {food101_root}')
    food101_folders = {
        f.lower().replace(' ', '_').replace('-', '_'): f
        for f in os.listdir(food101_root)
        if os.path.isdir(os.path.join(food101_root, f))
    }
    for key, canonical in FOOD101_MAP.items():
        if canonical in DROP_CLASSES:
            continue
        matched = next((v for k, v in food101_folders.items()
                        if key in k or k in key), None)
        if not matched:
            continue
        n = copy_images(os.path.join(food101_root, matched),
                        os.path.join(FILTERED_DIR, canonical), 'food101')
        stats.setdefault(canonical, {'kaggle': 0, 'food101': 0, 'yours': 0})
        stats[canonical]['food101'] += n
else:
    print('Food-101 not found — only self-collected + Kaggle will be used')

# 3. Your self-collected photos (added LAST, no cap — these are gold)
if os.path.exists(YOUR_PHOTOS):
    print(f'Your photos found: {YOUR_PHOTOS}')
    for your_name, canonical in YOUR_NAME_MAP.items():
        if canonical in DROP_CLASSES:
            continue
        src = os.path.join(YOUR_PHOTOS, your_name)
        if not os.path.exists(src):
            continue
        dst = os.path.join(FILTERED_DIR, canonical)
        os.makedirs(dst, exist_ok=True)
        imgs = [f for f in os.listdir(src)
                if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))]
        for img in imgs:
            shutil.copy(os.path.join(src, img),
                        os.path.join(dst, f'yours_{img}'))
        stats.setdefault(canonical, {'kaggle': 0, 'food101': 0, 'yours': 0})
        stats[canonical]['yours'] += len(imgs)
else:
    print(f'Your photos not found at {YOUR_PHOTOS}')

# ── Drop classes with fewer than 20 images (too few to train) ─────────────────
removed = []
for cls in os.listdir(FILTERED_DIR):
    cls_path = os.path.join(FILTERED_DIR, cls)
    n = len([f for f in os.listdir(cls_path)
             if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))])
    if n < 20:
        shutil.rmtree(cls_path)
        removed.append(f'{cls} ({n} imgs)')

if removed:
    print(f'Dropped (too few images): {removed}')

# ── Final summary ──────────────────────────────────────────────────────────────
CLASSES = sorted(os.listdir(FILTERED_DIR))
NUM_CLASSES = len(CLASSES)

print(f"\n{'Class':<25} {'Kaggle':>8} {'F-101':>7} {'Yours':>7} {'Total':>7}  Status")
print('─' * 72)
for cls in CLASSES:
    s = stats.get(cls, {})
    kg = s.get('kaggle', 0)
    f1 = s.get('food101', 0)
    yo = s.get('yours', 0)
    t  = kg + f1 + yo
    status = 'OK' if t >= 60 else ('LOW' if t >= 30 else 'VERY LOW')
    star = ' *KIIT*' if yo > 0 else ''
    print(f'  {cls:<25} {kg:>8} {f1:>7} {yo:>7} {t:>7}  {status}{star}')
print('─' * 72)
print(f'\n{NUM_CLASSES} classes ready | Dataset: {FILTERED_DIR}')
print('*KIIT* = has your real mess photos')

## Step 2: Train / Val / Test Split (70 / 15 / 15)

In [ ]:
import random
from sklearn.model_selection import train_test_split

SPLIT_DIR = '/content/split_dataset'
TRAIN_DIR = os.path.join(SPLIT_DIR, 'train')
VAL_DIR   = os.path.join(SPLIT_DIR, 'val')
TEST_DIR  = os.path.join(SPLIT_DIR, 'test')

if os.path.exists(SPLIT_DIR):
    shutil.rmtree(SPLIT_DIR)
for d in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
    os.makedirs(d, exist_ok=True)

random.seed(42)
np.random.seed(42)

print(f"{'Class':<25} {'Train':>7} {'Val':>6} {'Test':>6} {'Total':>7}")
print('─' * 55)

for cls in CLASSES:
    cls_path = os.path.join(FILTERED_DIR, cls)
    images = [f for f in os.listdir(cls_path)
              if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))]
    random.shuffle(images)

    train_imgs, temp = train_test_split(images, test_size=0.30, random_state=42)
    val_imgs, test_imgs = train_test_split(temp, test_size=0.50, random_state=42)

    for split, imgs in [('train', train_imgs), ('val', val_imgs), ('test', test_imgs)]:
        dst = os.path.join(SPLIT_DIR, split, cls)
        os.makedirs(dst, exist_ok=True)
        for img in imgs:
            shutil.copy(os.path.join(cls_path, img), os.path.join(dst, img))

    print(f'  {cls:<25} {len(train_imgs):>7} {len(val_imgs):>6} {len(test_imgs):>6} {len(images):>7}')

print('─' * 55)
total_imgs = sum(len(os.listdir(os.path.join(cls_p, cls)))
                 for cls in CLASSES
                 for cls_p in [FILTERED_DIR])
print(f'Split complete. Total images: {total_imgs}')

## Step 3: GPU Check

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU — training will be very slow. Go to Runtime > Change runtime type > T4 GPU')

## Step 4: Augmentation & DataLoaders

Food-safe augmentation rules:
- NO `RandomVerticalFlip` — food is never upside down
- NO `GaussianBlur` — destroys texture signals (poha, dosa, ghuguni)
- Rotation max 15° — food isn't tilted much in real photos
- Hue jitter max 0.05 — color is a key signal for many dishes

In [ ]:
import torchvision.transforms as T
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

IMG_SIZE   = 224
BATCH_SIZE = 32   # safe for T4 at 224x224

train_transform = T.Compose([
    T.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    T.RandomCrop(IMG_SIZE),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(degrees=15),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    T.RandomPerspective(distortion_scale=0.2, p=0.3),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])

eval_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])

train_dataset = ImageFolder(TRAIN_DIR, transform=train_transform)
val_dataset   = ImageFolder(VAL_DIR,   transform=eval_transform)
test_dataset  = ImageFolder(TEST_DIR,  transform=eval_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f'Train: {len(train_dataset)} images | Val: {len(val_dataset)} | Test: {len(test_dataset)}')
print(f'Classes: {NUM_CLASSES}')

# Verify class order matches CLASSES list
assert train_dataset.classes == CLASSES, 'Class order mismatch! Re-run Step 1.'
print('Class order verified OK')

## Step 5: Class-Weighted Loss

Inverse-frequency weighting so rare classes (ghuguni, dahibara) get more loss signal.

In [ ]:
class_counts = np.bincount([label for _, label in train_dataset.samples])
class_weights = 1.0 / class_counts.astype(float)
class_weights = class_weights / class_weights.sum() * NUM_CLASSES  # normalize
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

criterion = nn.CrossEntropyLoss(
    weight=class_weights_tensor,
    label_smoothing=0.1   # reduces overconfidence, improves generalization
)

print('Class weights (higher = rarer class):')
for i, (cls, cnt, wt) in enumerate(zip(CLASSES, class_counts, class_weights)):
    print(f'  {cls:<25} {cnt:>4} imgs  weight={wt:.3f}')

## Step 6: Model

In [ ]:
import timm

def build_model(num_classes, drop_path_rate=0.1):
    model = timm.create_model(
        'convnext_tiny',
        pretrained=True,
        num_classes=num_classes,
        drop_path_rate=drop_path_rate,
    )
    return model.to(device)

model = build_model(NUM_CLASSES)
total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'ConvNeXt-Tiny | {total_params:.1f}M params | {NUM_CLASSES} classes')

## Step 7: Training Utilities (MixUp + LLRD + Scheduler)

In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
import time

# ── MixUp ─────────────────────────────────────────────────────────────────────
def mixup_batch(images, labels, alpha=0.4):
    """Blend two images + interpolate their labels. Applied to 50% of batches."""
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(images.size(0)).to(device)
    return lam * images + (1 - lam) * images[idx], labels, labels[idx], lam

def mixup_criterion(criterion, pred, ya, yb, lam):
    return lam * criterion(pred, ya) + (1 - lam) * criterion(pred, yb)

# ── Layer-wise LR decay (LLRD) — earlier layers get smaller LR ────────────────
def get_llrd_optimizer(model, base_lr=1e-4, decay=0.65):
    # Track which parameters have already been assigned
    assigned = set()
    param_groups = []

    # ConvNeXt stages — stages.3 gets base_lr, earlier stages get decayed LR
    for stage_idx in range(4):
        lr = base_lr * (decay ** (3 - stage_idx))
        params = []
        for n, p in model.named_parameters():
            if f'stages.{stage_idx}' in n and p.requires_grad and id(p) not in assigned:
                params.append(p)
                assigned.add(id(p))
        if params:
            param_groups.append({'params': params, 'lr': lr})
            print(f'  stages.{stage_idx}: lr={lr:.2e} ({len(params)} tensors)')

    # Everything else (head, norm, stem) gets base_lr
    remaining = []
    for n, p in model.named_parameters():
        if p.requires_grad and id(p) not in assigned:
            remaining.append(p)
            assigned.add(id(p))
    if remaining:
        param_groups.append({'params': remaining, 'lr': base_lr})
        print(f'  head/norm/stem: lr={base_lr:.2e} ({len(remaining)} tensors)')

    return optim.AdamW(param_groups, weight_decay=1e-2)

# ── Freeze/unfreeze helpers ────────────────────────────────────────────────────
def freeze_backbone(model):
    for param in model.parameters():
        param.requires_grad = False
    for name, param in model.named_parameters():
        if 'head' in name or 'norm' in name:
            param.requires_grad = True
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  Head only | Trainable: {trainable/1e6:.2f}M params')

def unfreeze_stages(model, stages):
    for name, param in model.named_parameters():
        for s in stages:
            if f'stages.{s}' in name:
                param.requires_grad = True
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  Stages {stages} + head | Trainable: {trainable/1e6:.2f}M params')

def unfreeze_all(model):
    for param in model.parameters():
        param.requires_grad = True
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  Full model | Trainable: {trainable/1e6:.2f}M params')

# ── Single epoch runner ────────────────────────────────────────────────────────
def run_epoch(model, loader, criterion, optimizer=None, use_mixup=False):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, correct, total = 0.0, 0, 0

    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            if is_train and use_mixup and np.random.rand() < 0.5:
                images, ya, yb, lam = mixup_batch(images, labels)
                out = model(images)
                loss = mixup_criterion(criterion, out, ya, yb, lam)
            else:
                out = model(images)
                loss = criterion(out, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            total_loss += loss.item() * images.size(0)
            preds = out.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return total_loss / total, 100.0 * correct / total

# ── Balanced accuracy ──────────────────────────────────────────────────────────
def balanced_accuracy(model, loader):
    from sklearn.metrics import balanced_accuracy_score
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            preds = model(images).argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    return balanced_accuracy_score(all_labels, all_preds) * 100

print('Training utilities ready')

## Step 8: 3-Stage Training

```
Stage 1 (8 epochs):  Head only, LR=1e-2       — quick feature alignment
Stage 2 (7 epochs):  Unfreeze stages 2+3      — careful fine-tuning
Stage 3 (30 epochs): Full model + LLRD        — final convergence
```
Total: ~45–55 min on T4

In [ ]:
import json

STAGE1_EPOCHS = 8
STAGE2_EPOCHS = 7
STAGE3_EPOCHS = 30   # increased from 25 — model wasn't converged at 25
BASE_LR       = 1e-4
PATIENCE      = 10   # early stopping patience

best_val_bal_acc = 0.0
no_improve       = 0
history          = []
CKPT_PATH        = os.path.join(SAVE_DIR, 'convnext_tiny_best.pth')

def save_checkpoint(model, epoch, val_bal_acc, val_acc):
    torch.save({
        'epoch':            epoch,
        'model_state_dict': model.state_dict(),
        'val_bal_acc':      val_bal_acc,
        'val_acc':          val_acc,
        'classes':          CLASSES,
        'num_classes':      NUM_CLASSES,
    }, CKPT_PATH)

start_total = time.time()

# ── Stage 1: Head only ─────────────────────────────────────────────────────────
print('=' * 60)
print(f'STAGE 1: Head only ({STAGE1_EPOCHS} epochs, LR=1e-2)')
print('=' * 60)
freeze_backbone(model)
opt1 = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                   lr=1e-2, weight_decay=1e-2)
sch1 = CosineAnnealingLR(opt1, T_max=STAGE1_EPOCHS, eta_min=1e-4)

for ep in range(1, STAGE1_EPOCHS + 1):
    t_loss, t_acc = run_epoch(model, train_loader, criterion, opt1, use_mixup=False)
    v_loss, v_acc = run_epoch(model, val_loader,   criterion)
    v_bal = balanced_accuracy(model, val_loader)
    sch1.step()
    history.append({'stage': 1, 'ep': ep, 'v_bal': v_bal, 'v_acc': v_acc})
    if v_bal > best_val_bal_acc:
        best_val_bal_acc = v_bal
        save_checkpoint(model, ep, v_bal, v_acc)
    print(f'  Ep {ep:02d} | loss={t_loss:.3f} | train_acc={t_acc:.1f}% | val_acc={v_acc:.1f}% | val_bal={v_bal:.1f}%  {"<< best" if v_bal == best_val_bal_acc else ""}')

# ── Stage 2: Unfreeze stages 2+3 ──────────────────────────────────────────────
print()
print('=' * 60)
print(f'STAGE 2: Stages 2+3 unfrozen ({STAGE2_EPOCHS} epochs, LR=5e-4)')
print('Note: accuracy will DROP at epoch 9 — this is normal!')
print('=' * 60)
unfreeze_stages(model, [2, 3])
opt2 = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                   lr=5e-4, weight_decay=1e-2)
sch2 = CosineAnnealingLR(opt2, T_max=STAGE2_EPOCHS, eta_min=1e-5)

for ep in range(1, STAGE2_EPOCHS + 1):
    t_loss, t_acc = run_epoch(model, train_loader, criterion, opt2, use_mixup=False)
    v_loss, v_acc = run_epoch(model, val_loader,   criterion)
    v_bal = balanced_accuracy(model, val_loader)
    sch2.step()
    history.append({'stage': 2, 'ep': ep, 'v_bal': v_bal, 'v_acc': v_acc})
    if v_bal > best_val_bal_acc:
        best_val_bal_acc = v_bal
        no_improve = 0
        save_checkpoint(model, STAGE1_EPOCHS + ep, v_bal, v_acc)
    print(f'  Ep {ep:02d} | loss={t_loss:.3f} | train_acc={t_acc:.1f}% | val_acc={v_acc:.1f}% | val_bal={v_bal:.1f}%  {"<< best" if v_bal == best_val_bal_acc else ""}')

# ── Stage 3: Full model + LLRD ─────────────────────────────────────────────────
print()
print('=' * 60)
print(f'STAGE 3: Full model + LLRD ({STAGE3_EPOCHS} epochs, base_lr={BASE_LR})')
print('=' * 60)
unfreeze_all(model)
opt3 = get_llrd_optimizer(model, base_lr=BASE_LR, decay=0.65)
sch3 = CosineAnnealingLR(opt3, T_max=STAGE3_EPOCHS, eta_min=1e-6)

for ep in range(1, STAGE3_EPOCHS + 1):
    t_loss, t_acc = run_epoch(model, train_loader, criterion, opt3, use_mixup=True)
    v_loss, v_acc = run_epoch(model, val_loader,   criterion)
    v_bal = balanced_accuracy(model, val_loader)
    sch3.step()
    history.append({'stage': 3, 'ep': ep, 'v_bal': v_bal, 'v_acc': v_acc})

    improved = v_bal > best_val_bal_acc
    if improved:
        best_val_bal_acc = v_bal
        no_improve = 0
        save_checkpoint(model, STAGE1_EPOCHS + STAGE2_EPOCHS + ep, v_bal, v_acc)
    else:
        no_improve += 1

    tag = '<< best' if improved else f'(no improve {no_improve}/{PATIENCE})'
    print(f'  Ep {ep:02d} | loss={t_loss:.3f} | train_acc={t_acc:.1f}% | val_acc={v_acc:.1f}% | val_bal={v_bal:.1f}%  {tag}')

    if no_improve >= PATIENCE:
        print(f'  Early stopping at stage 3 epoch {ep}')
        break

elapsed = (time.time() - start_total) / 60
print()
print(f'Training complete in {elapsed:.1f} min')
print(f'Best val balanced accuracy: {best_val_bal_acc:.2f}%')
print(f'Checkpoint saved: {CKPT_PATH}')

## Step 9: Final Test Evaluation

This is the REAL accuracy number. Val accuracy is inflated because we picked the best checkpoint on val.

In [ ]:
from sklearn.metrics import classification_report, balanced_accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Load best checkpoint
ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'Loaded checkpoint from epoch {ckpt["epoch"]} (val_bal_acc={ckpt["val_bal_acc"]:.2f}%)')

# Run on test set
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        preds = model(images).argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

test_bal_acc  = balanced_accuracy_score(all_labels, all_preds) * 100
test_flat_acc = 100.0 * sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)

print(f'\nTEST BALANCED ACCURACY: {test_bal_acc:.2f}%  (primary metric)')
print(f'TEST FLAT ACCURACY:     {test_flat_acc:.2f}%')
print()
print(classification_report(all_labels, all_preds, target_names=CLASSES))

# Confusion matrix
if NUM_CLASSES <= 20:
    cm = confusion_matrix(all_labels, all_preds)
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASSES, yticklabels=CLASSES, ax=ax)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title('Confusion Matrix (Test Set)')
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, 'confusion_matrix.png'), dpi=100)
    plt.show()
    print('Confusion matrix saved to Drive')

## Step 10: Inference Function with Confidence Threshold

- Returns top-3 predictions with confidence %
- If top-1 confidence < threshold → returns a message asking user to type manually
- Uses Test-Time Augmentation (TTA) for better accuracy

In [ ]:
from PIL import Image

# ── Config ─────────────────────────────────────────────────────────────────────
CONFIDENCE_THRESHOLD = 62.5   # tune this based on real-world testing
                               # if top-1 < this, ask user to type manually

# TTA transforms — slightly varied views of the same image
tta_transforms = [
    eval_transform,
    T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.RandomHorizontalFlip(p=1.0),
               T.ToTensor(), T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])]),
    T.Compose([T.Resize((IMG_SIZE+20, IMG_SIZE+20)), T.CenterCrop(IMG_SIZE),
               T.ToTensor(), T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])]),
    T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.RandomRotation(10),
               T.ToTensor(), T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])]),
]

def predict_dish(image_input, top_k=3, use_tta=True):
    """
    Predict food from image with confidence threshold.

    Args:
        image_input : file path (str) or PIL Image
        top_k       : number of predictions to return
        use_tta     : use Test-Time Augmentation (recommended)

    Returns dict with:
        status       : 'ok' or 'low_confidence'
        predictions  : list of {dish, confidence} top-k
        message      : user-facing message
    """
    model.eval()

    if isinstance(image_input, str):
        img = Image.open(image_input).convert('RGB')
    else:
        img = image_input.convert('RGB')

    with torch.no_grad():
        if use_tta:
            logits_list = []
            for tf in tta_transforms:
                try:
                    tensor = tf(img).unsqueeze(0).to(device)
                    logits_list.append(model(tensor))
                except Exception:
                    pass
            logits = torch.stack(logits_list).mean(dim=0)
        else:
            tensor = eval_transform(img).unsqueeze(0).to(device)
            logits = model(tensor)

    probs = F.softmax(logits, dim=1)[0]
    top_probs, top_idx = probs.topk(min(top_k, NUM_CLASSES))

    predictions = [
        {
            'dish':       CLASSES[idx.item()].replace('_', ' ').title(),
            'class_id':   idx.item(),
            'confidence': round(prob.item() * 100, 1)
        }
        for prob, idx in zip(top_probs, top_idx)
    ]

    top_confidence = predictions[0]['confidence']

    if top_confidence >= CONFIDENCE_THRESHOLD:
        return {
            'status':      'ok',
            'predictions': predictions,
            'message':     f'Detected: {predictions[0]["dish"]} ({top_confidence}% confidence)',
        }
    else:
        return {
            'status':      'low_confidence',
            'predictions': predictions,  # still return top-3 so user can pick
            'message':     (
                f'Could not identify this food with enough confidence '
                f'(best guess: {predictions[0]["dish"]} at {top_confidence}%). '
                f'Please type the food name manually or select from the suggestions below.'
            ),
        }

print(f'Inference ready | Confidence threshold: {CONFIDENCE_THRESHOLD}%')
print('Usage: result = predict_dish("path/to/image.jpg")')

## Step 11: Test the Inference Function

Upload a food photo and test it here.

In [ ]:
import random
import matplotlib.pyplot as plt

# ── Test on random images from test set ───────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

sampled = []
for cls in random.sample(CLASSES, min(8, NUM_CLASSES)):
    cls_path = os.path.join(TEST_DIR, cls)
    imgs = [f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg','.jpeg','.png'))]
    if imgs:
        sampled.append((cls, os.path.join(cls_path, random.choice(imgs))))

for i, (true_cls, img_path) in enumerate(sampled):
    result = predict_dish(img_path, top_k=3, use_tta=True)
    pred = result['predictions'][0]
    correct = pred['class_id'] == CLASSES.index(true_cls)

    img = Image.open(img_path)
    axes[i].imshow(img)
    axes[i].axis('off')

    color = 'green' if correct else 'red'
    title  = f'True: {true_cls.replace("_"," ")}\n'
    title += f'Pred: {pred["dish"]} ({pred["confidence"]}%)'
    if result['status'] == 'low_confidence':
        title += '\n[LOW CONF]'
    axes[i].set_title(title, color=color, fontsize=9)

plt.suptitle(f'Sample Predictions (Threshold: {CONFIDENCE_THRESHOLD}%)', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'sample_predictions.png'), dpi=100, bbox_inches='tight')
plt.show()
print('Sample predictions saved to Drive')

## Done!

**Files in Drive/FitAI/models_v4/:**
- `convnext_tiny_best.pth` — trained model
- `classes.json` — class list + config
- `nutrition_db.json` — calorie/macro data
- `main.py` — FastAPI server (give to backend)
- `requirements.txt` — dependencies
- `confusion_matrix.png` — visual diagnosis
- `sample_predictions.png` — sanity check

**Tuning the threshold:**
- If model gives too many 'low confidence' responses → lower threshold
- If model gives wrong confident predictions → raise threshold